# NCCT Stroke Segmentation — Kaggle (2× T4) with Imbalance Mitigation

Segment ischemic stroke regions from Non-Contrast CT (NCCT) brain images using [MMSegmentation](https://github.com/open-mmlab/mmsegmentation).

**Key improvements vs baseline:**
- **Multi-loss strategy**: CrossEntropyLoss (weighted) + DiceLoss — tackles 98:2 class imbalance
- **Proper inverse-frequency weighting**: `class_weight=[0.05, 1.0]` (20× emphasis on stroke)
- **Enhanced augmentations**: RandomRotate for head tilt variation
- **Multi-GPU DDP training**: Uses both T4 GPUs via `torchrun`
- **GPU-optimized batch size**: batch=24 per GPU → 48 effective (maximizes T4 memory)
- **Test visualization**: Per-class metrics + side-by-side predictions

---
**Runtime**: Kaggle with 2× T4 GPU or any 2-GPU system  
**Storage needed**: ~12 GB  
**Time**: ~1-2 hours for full 20K schedule (batch 24 per GPU)

## 1. Environment Setup

Kaggle provides 2× T4 GPUs with PyTorch pre-installed.

> **Important**: Before running, make sure to:
> 1. Set **Accelerator → GPU T4 x2** in Kaggle Notebook settings
> 2. Add Internet access: **Session options → Internet = ON**
> 3. (Optional) Upload the NCCT dataset as a Kaggle Dataset for faster access

In [ ]:
import torch, os, sys, subprocess, json, glob, warnings, shutil, time
import numpy as np
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

print(f"\nKaggle env: NPROC_PER_NODE={os.environ.get('NPROC_PER_NODE', 'not set')}")

### Install Dependencies

MMSegmentation requires:
- **MMCV**: Pre-built CUDA wheel (community build)
- **MMEngine**: Via pip
- **Project**: Install the forked repo in editable mode

In [ ]:
torch_ver = torch.__version__.split("+")[0]
cuda_ver = torch.version.cuda
cuda_short = cuda_ver.replace(".", "")
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"torch={torch_ver}  cuda={cuda_ver}  python={sys.version_info.major}.{sys.version_info.minor}")

!pip install -q gdown

# Uninstall mmcv-lite if previously installed
!pip uninstall mmcv-lite -y -q 2>/dev/null || true

# ---- Install MMCV ----
RELEASE_TAG = "mmcv-2.2.0%2Ba8073c7"
WHEEL_FILE = f"mmcv-2.2.0%2Ba8073c7pt{torch_ver}cu{cuda_short}-{py_ver}-{py_ver}-linux_x86_64.whl"
WHEEL_URL = f"https://github.com/MiroPsota/torch_packages_builder/releases/download/{RELEASE_TAG}/{WHEEL_FILE}"
WHEEL_PATH = f"/tmp/{WHEEL_FILE}"
print(f"Wheel: pt{torch_ver}cu{cuda_short} {py_ver}")

MMCV_INSTALLED = False

if not os.path.exists(WHEEL_PATH):
    ret = os.system(f"curl -L --retry 5 --retry-delay 10 --connect-timeout 30 "
                    f"'{WHEEL_URL}' -o '{WHEEL_PATH}' 2>&1 | tail -3")
else:
    ret = 0
    print("Wheel already downloaded.")

if ret == 0 and os.path.exists(WHEEL_PATH) and os.path.getsize(WHEEL_PATH) > 1_000_000:
    print("Installing MMCV from downloaded wheel...")
    r = subprocess.run(["pip", "install", WHEEL_PATH], capture_output=True, text=True)
    if r.returncode == 0:
        print("MMCV installed from community wheel.")
        MMCV_INSTALLED = True
    else:
        print(f"pip install failed: {r.stderr[-200:]}")
        os.remove(WHEEL_PATH)

if not MMCV_INSTALLED:
    print("Community wheel unavailable. Building MMCV from source...")
    print("This takes 10-15 minutes.")
    !pip install "mmcv>=2.0.0rc4,<2.5.0" -q
    print("MMCV installed via pip.")

!pip install -q mmengine

import mmcv
print(f"mmcv: {mmcv.__version__}")
print(f"mmcv.ops available: {hasattr(mmcv, 'ops')}")

In [ ]:
# Clone the repository
REPO_URL = "https://github.com/lhfazry/ncct-segmentation"
REPO_DIR = "/kaggle/working/ncct-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git status --short

In [ ]:
# Install the project and extra dependencies
!pip install -e . -q
!pip install ftfy regex albumentations pandas scikit-learn -q

# Patch mmseg/__init__.py: relax MMCV version check
import re
with open(f"{REPO_DIR}/mmseg/__init__.py") as f:
    _init_code = f.read()
_init_code = _init_code.replace(
    "MMCV_MAX = '2.3.0'",
    "MMCV_MAX = '3.0.0'")
_init_code = re.sub(
    r'assert \(mmcv_min_version <= mmcv_version < mmcv_max_version\),.*?'
    r'f\'Please install mmcv>=2\.0\.0rc4\.\'',
    """if not (mmcv_min_version <= mmcv_version < mmcv_max_version):
    import warnings
    warnings.warn(
        f'MMCV=={mmcv.__version__} is used but may be incompatible. '
        f'Expected mmcv>={MMCV_MIN}, <{MMCV_MAX}.')""",
    _init_code,
    flags=re.DOTALL,
)
with open(f"{REPO_DIR}/mmseg/__init__.py", "w") as f:
    f.write(_init_code)
print("Patched mmseg/__init__.py: MMCV version assert -> warning")

# Patch mmseg/models/backbones/mit.py: fix Version comparison
mit_path = f"{REPO_DIR}/mmseg/models/backbones/mit.py"
with open(mit_path) as f:
    mit_code = f.read()
mit_code = mit_code.replace(
    "import math\nimport warnings",
    "import math\nimport warnings\n\nfrom packaging.version import Version"
)
mit_code = mit_code.replace(
    "if mmcv_version < digit_version('1.3.17'):",
    "if mmcv_version < Version('1.3.17'):"
)
with open(mit_path, "w") as f:
    f.write(mit_code)
print("Patched mmseg/models/backbones/mit.py: Version comparison fix")

from mmseg.utils import register_all_modules
register_all_modules()

import mmseg
print(f"mmseg version: {mmseg.__version__}")

---
## 2. Dataset Preparation

### Option A: Download from Google Drive (requires Internet)
### Option B: Use Kaggle Dataset (faster, recommended)

If you've uploaded the NCCT dataset as a Kaggle dataset, set `KAGGLE_DATASET_PATH` below.

In [ ]:
# ---- Configuration ----
# Set this if you uploaded the dataset to Kaggle Datasets:
# Example: KAGGLE_DATASET_PATH = "/kaggle/input/ncct-stroke-dataset"
KAGGLE_DATASET_PATH = None  # Set to None to download from Drive

DATA_ROOT = "/kaggle/working/data/ncct"
os.makedirs(DATA_ROOT, exist_ok=True)

if KAGGLE_DATASET_PATH and os.path.exists(KAGGLE_DATASET_PATH):
    print(f"Using Kaggle Dataset from: {KAGGLE_DATASET_PATH}")
    !cp -r "{KAGGLE_DATASET_PATH}"/* /kaggle/working/data/
else:
    print("Downloading dataset from Google Drive...")
    import gdown
    import zipfile
    FILE_ID = "1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo"
    ZIP_PATH = "/kaggle/working/dataset.zip"
    EXTRACT_DIR = "/kaggle/working/dataset_raw"

    if not os.path.exists(ZIP_PATH):
        url = f"https://drive.google.com/uc?id={FILE_ID}"
        gdown.download(url, ZIP_PATH, quiet=False)

    if not os.path.exists(EXTRACT_DIR):
        print("Extracting dataset...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("Extraction complete.")

    # Organize into MMSegmentation structure
    for split in ["train", "val", "test"]:
        src_img = os.path.join(EXTRACT_DIR, split, "images")
        src_mask = os.path.join(EXTRACT_DIR, split, "masks")
        if os.path.isdir(src_img) and os.path.isdir(src_mask):
            dst_img = os.path.join(DATA_ROOT, split, "images")
            dst_mask = os.path.join(DATA_ROOT, split, "masks")
            os.makedirs(os.path.dirname(dst_img), exist_ok=True)
            if not os.path.exists(dst_img):
                shutil.copytree(src_img, dst_img)
            if not os.path.exists(dst_mask):
                shutil.copytree(src_mask, dst_mask)

    # Clean up zip to save space
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)

print(f"Dataset at: {DATA_ROOT}")
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATA_ROOT, split, "images")
    mask_dir = os.path.join(DATA_ROOT, split, "masks")
    n_imgs = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    n_masks = len(os.listdir(mask_dir)) if os.path.isdir(mask_dir) else 0
    print(f"  {split}: {n_imgs} images, {n_masks} masks")

---
## 3. Class Balance Analysis

Understanding the exact class distribution helps tune the loss weights.

In [ ]:
from PIL import Image

mask_dir = os.path.join(DATA_ROOT, "train", "masks")
total_px = 0
stroke_px = 0

mask_files = sorted(os.listdir(mask_dir))
for fname in mask_files:
    m = np.array(Image.open(os.path.join(mask_dir, fname)).convert("L"))
    total_px += m.size
    stroke_px += (m > 127).sum()

bg_pct = (total_px - stroke_px) / total_px * 100
stroke_pct = stroke_px / total_px * 100
ratio = bg_pct / stroke_pct

print(f"Class distribution (train):")
print(f"  Background: {bg_pct:.2f}%")
print(f"  Stroke:     {stroke_pct:.2f}%")
print(f"  Ratio:      {ratio:.1f}:1 (bg:stroke)")

# Compute inverse-frequency weights
n_classes = 2
inv_weight_bg = 100 / (n_classes * bg_pct)
inv_weight_stroke = 100 / (n_classes * stroke_pct)
print(f"\nInverse-frequency weights: [{inv_weight_bg:.4f}, {inv_weight_stroke:.4f}]")
print(f"Normalized (stroke=1.0): [{inv_weight_bg/inv_weight_stroke:.4f}, 1.0]")

# Find images with highest stroke ratio (for analysis)
stroke_ratios = []
for fname in mask_files[:500]:
    m = np.array(Image.open(os.path.join(mask_dir, fname)).convert("L"))
    sr = (m > 127).sum() / m.size * 100
    stroke_ratios.append(sr)

print(f"\nStroke ratio stats (first 500 samples):")
print(f"  Mean: {np.mean(stroke_ratios):.2f}%")
print(f"  Median: {np.median(stroke_ratios):.2f}%")
print(f"  Max: {np.max(stroke_ratios):.2f}%")
print(f"  Samples with 0% stroke: {sum(1 for s in stroke_ratios if s == 0)} / {len(stroke_ratios)}")

---
## 4. Configuration

The repo already has 6 architecture configs under `configs/ncct/` with:
- **Multi-loss**: CrossEntropyLoss (weighted [0.05, 1.0]) + DiceLoss
- **AMP**: Mixed precision training
- **Enhanced augmentations**: RandomResize, RandomFlip, RandomRotate

We'll patch `data_root` for the Kaggle environment and pick the architecture.

In [ ]:
from mmengine.config import Config

# Pick your architecture
# Options: unet_fcn, unet_deeplabv3, unet_pspnet, deeplabv3plus_r50, segformer_mitb0, pspnet_r50
ARCH_NAME = "unet_fcn"  # <-- CHANGE THIS to try different architectures

CFG_FILE = f"{REPO_DIR}/configs/ncct/{ARCH_NAME}_stroke_ncct.py"
WORK_DIR = f"/kaggle/working/work_dirs/{ARCH_NAME}"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Architecture: {ARCH_NAME}")
print(f"Config: {CFG_FILE}")
print(f"Work dir: {WORK_DIR}")

# Load and patch for Kaggle
cfg = Config.fromfile(CFG_FILE)
_abs_root = DATA_ROOT + "/"
cfg.data_root = _abs_root
for _dl_key in ["train_dataloader", "val_dataloader", "test_dataloader"]:
    if hasattr(cfg, _dl_key) and hasattr(getattr(cfg, _dl_key), "dataset"):
        getattr(cfg, _dl_key).dataset.data_root = _abs_root

cfg.work_dir = WORK_DIR

# Dump patched config
PATCHED_CFG = f"/kaggle/working/patched_config.py"
cfg.dump(PATCHED_CFG)
print(f"Patched config dumped to: {PATCHED_CFG}")

# Verify the loss config
dec_loss = cfg.model['decode_head']['loss_decode']
print(f"\nDecode head losses: {len(dec_loss) if isinstance(dec_loss, list) else 1}")
if isinstance(dec_loss, list):
    for l in dec_loss:
        print(f"  - {l['type']} (weight={l.get('loss_weight', 1.0)})")
        if 'class_weight' in l:
            print(f"    class_weight={l['class_weight']}")
            
# Print current batch size setting
bs = cfg.train_dataloader.batch_size
nw = cfg.train_dataloader.num_workers
print(f"\nTrain batch_size per GPU: {bs}")
print(f"Effective batch (2 GPUs):  {bs * 2}")
print(f"Num workers: {nw}")

---
## 5. Multi-GPU Training

Using `torchrun` for Distributed Data Parallel (DDP) on 2× T4 GPUs.

**GPU optimization strategy:**
- **batch=24 per GPU** → effective batch 48 (config default, fills ~13 GB / 16 GB T4)
- **num_workers=4** per dataloader → keeps both GPUs fed without CPU bottleneck
- **AMP (mixed precision)** → 2× memory efficiency vs FP32, enables larger batches
- **NCCL backend** (default for torchrun) → fastest GPU-GPU communication
- **cuDNN benchmark** auto-tunes kernels on first iteration
- **persistent_workers=True** → avoids worker spawn overhead each epoch

> If you hit OOM (e.g., DeepLabV3+ or PSPNet R-50), override:
> `--cfg-options train_dataloader.batch_size=16`

**Training schedule:**
- 20,000 iterations
- Validation every 2,000 iterations
- Mixed precision (AMP)

In [ ]:
# Quick GPU check before training
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | Total memory: {props.total_memory / 1e9:.1f} GB")

# Run nvidia-smi
!nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu --format=csv,noheader

> **Tip**: To monitor GPU during training, run `!nvidia-smi -l 2` in a new cell.

In [ ]:
%%time

# ---- GPU Optimization Options ----
# The config defaults to batch=24 per GPU (48 effective on 2 GPUs).
# Adjust BATCH_SIZE if you hit OOM with larger architectures:
#   U-Net variants:   24 works comfortably
#   DeepLabV3+ R-50:  16 recommended
#   PSPNet R-50:      16 recommended
#   SegFormer MIT-B0: 32 works comfortably
BATCH_SIZE = None  # Set to int to override config, e.g. BATCH_SIZE = 16

import subprocess

NUM_GPUS = torch.cuda.device_count()
print(f"Starting DDP training on {NUM_GPUS} GPU(s)...")
print(f"Architecture: {ARCH_NAME}")
print(f"Config: {PATCHED_CFG}")
print(f"Work dir: {WORK_DIR}")

# Build --cfg-options
_cfg_opts = [f"data_root={_abs_root}"]
if BATCH_SIZE is not None:
    _cfg_opts.append(f"train_dataloader.batch_size={BATCH_SIZE}")
    print(f"Batch size (per GPU): {BATCH_SIZE}  (effective: {BATCH_SIZE * NUM_GPUS})")
else:
    print(f"Batch size (per GPU): from config (effective: config_val × {NUM_GPUS} GPUs)")

if NUM_GPUS > 1:
    # Use torchrun for multi-GPU DDP
    # --nproc_per_node: GPUs per node
    # --master_port: avoid port conflicts (randomized)
    # --launcher=pytorch: use torch.distributed.launch protocol
    import random
    _master_port = random.randint(12000, 29000)
    cmd = [
        "torchrun", f"--nproc_per_node={NUM_GPUS}",
        f"--master_port={_master_port}",
        f"{REPO_DIR}/tools/train.py", PATCHED_CFG,
        f"--work-dir={WORK_DIR}", "--amp",
        f"--launcher=pytorch",
        "--cfg-options", * _cfg_opts,
    ]
else:
    # Single GPU fallback
    cmd = [
        "python", f"{REPO_DIR}/tools/train.py", PATCHED_CFG,
        f"--work-dir={WORK_DIR}", "--amp",
        "--cfg-options", * _cfg_opts,
    ]

print(f"\nCommand: {' '.join(cmd)}")
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
print(f"\nTraining completed with return code: {result.returncode if hasattr(result, 'returncode') else 'N/A'}")

---
## 6. Collect Validation Metrics

Parse the training log to extract best validation mDice and mIoU.

In [ ]:
def extract_val_metrics(work_dir):
    log_files = sorted(glob.glob(os.path.join(work_dir, "*.log.json")))
    if not log_files:
        return None
    val_metrics = []
    with open(log_files[-1]) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            if entry.get("mode") == "val" and "mDice" in entry:
                val_metrics.append({
                    "iter": entry.get("iteration", 0),
                    "mDice": entry.get("mDice", 0),
                    "mIoU": entry.get("mIoU", 0),
                    "aAcc": entry.get("aAcc", 0),
                    "Dice/0": entry.get("Dice/0", None),
                    "Dice/1": entry.get("Dice/1", None),
                })
    return val_metrics

# Also collect per-class metrics (if available with newer mmseg)
def extract_perclass_metrics(work_dir):
    log_files = sorted(glob.glob(os.path.join(work_dir, "*.log.json")))
    if not log_files:
        return None
    metrics = []
    with open(log_files[-1]) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            if entry.get("mode") == "val":
                # Check for per-class Dice
                dices = {k: v for k, v in entry.items() if k.startswith("Dice/")}
                if dices:
                    dices["iter"] = entry.get("iteration", 0)
                    metrics.append(dices)
    return metrics


val_metrics = extract_val_metrics(WORK_DIR)
perclass_metrics = extract_perclass_metrics(WORK_DIR)

if val_metrics:
    best = max(val_metrics, key=lambda x: x["mDice"])
    print(f"Best validation mDice: {best['mDice']:.4f}  (iter {best['iter']})")
    print(f"Best validation mIoU:  {best['mIoU']:.4f}  (iter {best['iter']})")
    
    final = val_metrics[-1]
    print(f"\nFinal validation mDice: {final['mDice']:.4f}  (iter {final['iter']})")
    print(f"Final validation mIoU:  {final['mIoU']:.4f}")
    
    if perclass_metrics:
        print(f"\nPer-class Dice:")
        for pc in perclass_metrics:
            dice_str = "  ".join([f"{k}: {v:.4f}" for k, v in pc.items() if k != "iter"])
            print(f"  iter {pc['iter']}: {dice_str}")
else:
    print("No validation metrics found. Training may be incomplete.")
    print(f"Check logs in: {WORK_DIR}")

### Validation Curves

In [ ]:
import matplotlib.pyplot as plt

if val_metrics:
    iters = [m["iter"] for m in val_metrics]
    dices = [m["mDice"] for m in val_metrics]
    ious = [m["mIoU"] for m in val_metrics]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(iters, dices, "-o", color="#2196F3", linewidth=2)
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("mDice")
    axes[0].set_title("Validation mDice")
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(y=best["mDice"], color="green", linestyle="--", alpha=0.7, label=f"Best: {best['mDice']:.4f}")
    axes[0].legend()

    axes[1].plot(iters, ious, "-s", color="#FF5722", linewidth=2)
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("mIoU")
    axes[1].set_title("Validation mIoU")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "validation_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {WORK_DIR}/validation_curves.png")
else:
    print("No validation metrics to plot.")

---
## 7. Test Set Evaluation

Evaluate the best checkpoint on the held-out test set with per-class metrics.

In [ ]:
# Find latest checkpoint
ckpts = sorted(glob.glob(os.path.join(WORK_DIR, "iter_*.pth")))
if not ckpts:
    ckpts = sorted(glob.glob(os.path.join(WORK_DIR, "*.pth")))

if ckpts:
    BEST_CKPT = ckpts[-1]
    print(f"Best checkpoint: {os.path.basename(BEST_CKPT)}")
    print(f"Full path: {BEST_CKPT}")
else:
    BEST_CKPT = None
    print("No checkpoint found! Train the model first.")

In [ ]:
%%time
if BEST_CKPT:
    print("Running test evaluation...")
    !cd {REPO_DIR} && python tools/test.py {PATCHED_CFG} \
        {BEST_CKPT} \
        --show-dir {WORK_DIR}/test_preds \
        --out {WORK_DIR}/test_results.pkl \
        --cfg-options data_root={_abs_root}
else:
    print("No checkpoint. Skipping test evaluation.")

In [ ]:
# Parse test results
import pickle

results_pkl = f"{WORK_DIR}/test_results.pkl"
if os.path.exists(results_pkl):
    with open(results_pkl, "rb") as f:
        results = pickle.load(f)
    
    print("=" * 50)
    print(f"Test Results: {ARCH_NAME}")
    print("=" * 50)
    
    if hasattr(results, 'items'):
        for k, v in results.items():
            if isinstance(v, (int, float, np.floating)):
                print(f"  {k}: {v:.4f}")
            elif isinstance(v, np.ndarray):
                per_class = [f"{x:.4f}" for x in v.flatten()]
                print(f"  {k}: {per_class}")
            elif isinstance(v, dict):
                for sk, sv in v.items():
                    if isinstance(sv, np.ndarray):
                        print(f"  {k}/{sk}: {[f'{x:.4f}' for x in sv.flatten()]}")
                    else:
                        print(f"  {k}/{sk}: {sv:.4f}")
            else:
                print(f"  {k}: {v}")
    else:
        print(str(results))
else:
    print(f"Results file not found: {results_pkl}")
    print("Check if test evaluation completed successfully.")

---
## 8. Test Set Visualization

Visualize model predictions on test samples with:
- Input NCCT scan (grayscale)
- Ground truth mask (stroke in white)
- Predicted segmentation
- Prediction boundary overlaid on input
- Error map (FP in red, FN in blue)

In [ ]:
from mmseg.apis import init_model, inference_model
from scipy.ndimage import binary_dilation

if BEST_CKPT:
    # Load model
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = init_model(PATCHED_CFG, BEST_CKPT, device=device)

    # Test data
    test_img_dir = os.path.join(DATA_ROOT, "test/images")
    test_mask_dir = os.path.join(DATA_ROOT, "test/masks")
    test_images = sorted(os.listdir(test_img_dir))
    print(f"Total test images: {len(test_images)}")

    # Select a diverse sample (images with and without stroke)
    sample_count = min(8, len(test_images))
    
    # Try to pick samples with stroke first
    sample_files = []
    for fname in test_images:
        mask_path = os.path.join(test_mask_dir, fname)
        mask_np = np.array(Image.open(mask_path).convert("L"))
        has_stroke = (mask_np > 127).sum() > 0
        if has_stroke:
            sample_files.append(fname)
        if len(sample_files) >= sample_count:
            break
    
    # Fall back to first N if not enough stroke images
    if len(sample_files) < sample_count:
        sample_files = test_images[:sample_count]
    
    print(f"Visualizing {len(sample_files)} test samples...")

    # Collect per-image Dice scores
    per_image_dice = []

    fig, axes = plt.subplots(len(sample_files), 5, figsize=(20, 4 * len(sample_files)))
    if len(sample_files) == 1:
        axes = axes.reshape(1, -1)

    for i, fname in enumerate(sample_files):
        img_path = os.path.join(test_img_dir, fname)
        mask_path = os.path.join(test_mask_dir, fname)

        # Load inputs
        img_np = np.array(Image.open(img_path).convert("L"))
        mask_np = np.array(Image.open(mask_path).convert("L"))
        mask_bin = (mask_np > 127).astype(np.uint8)

        # Run inference
        result = inference_model(model, img_path)
        pred = result.pred_sem_seg.data.cpu().numpy()
        pred_bin = (pred[0] > 0).astype(np.uint8)

        # Compute per-image Dice
        intersection = (pred_bin & mask_bin).sum()
        union = pred_bin.sum() + mask_bin.sum()
        dice = 2 * intersection / union if union > 0 else 1.0
        per_image_dice.append(dice)

        # Error map: FP (pred=1, gt=0) in red, FN (pred=0, gt=1) in blue
        fp = (pred_bin == 1) & (mask_bin == 0)
        fn = (pred_bin == 0) & (mask_bin == 1)
        tp = (pred_bin == 1) & (mask_bin == 1)
        
        error_map = np.zeros((*img_np.shape, 3), dtype=np.uint8)
        error_map[..., 0] = fp.astype(np.uint8) * 255  # Red for FP
        error_map[..., 1] = tp.astype(np.uint8) * 255   # Green for TP
        error_map[..., 2] = fn.astype(np.uint8) * 255   # Blue for FN

        # Overlay: prediction outline on input
        outline = binary_dilation(pred_bin, iterations=1) ^ pred_bin
        overlay = np.stack([img_np] * 3, axis=-1).astype(np.float32)
        # Red outline for prediction
        overlay[:, :, 0] = np.where(outline, 255, overlay[:, :, 0])
        overlay[:, :, 1] = np.where(outline, 0, overlay[:, :, 1])
        overlay[:, :, 2] = np.where(outline, 0, overlay[:, :, 2])

        # Plot
        axes[i][0].imshow(img_np, cmap="gray")
        axes[i][0].set_title(f"Input NCCT\n{fname[:15]}...", fontsize=9)
        axes[i][0].axis("off")

        axes[i][1].imshow(mask_bin, cmap="gray")
        axes[i][1].set_title(f"Ground Truth\n(stroke=white)", fontsize=9)
        axes[i][1].axis("off")

        axes[i][2].imshow(pred_bin, cmap="gray")
        axes[i][2].set_title(f"Prediction\nDice={dice:.3f}", fontsize=9)
        axes[i][2].axis("off")

        axes[i][3].imshow(overlay.astype(np.uint8))
        axes[i][3].set_title("Overlay\n(pred edge in red)", fontsize=9)
        axes[i][3].axis("off")

        axes[i][4].imshow(error_map)
        axes[i][4].set_title("Error Map\nR=FP  G=TP  B=FN", fontsize=9)
        axes[i][4].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "test_visualization.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {WORK_DIR}/test_visualization.png")

    # Per-image Dice statistics
    per_image_dice = np.array(per_image_dice)
    print(f"\nPer-image Dice statistics (n={len(per_image_dice)}):")
    print(f"  Mean: {per_image_dice.mean():.4f}")
    print(f"  Median: {np.median(per_image_dice):.4f}")
    print(f"  Std: {per_image_dice.std():.4f}")
    print(f"  Min: {per_image_dice.min():.4f}")
    print(f"  Max: {per_image_dice.max():.4f}")

    # Histogram of per-image Dice
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(per_image_dice, bins=20, color="#2196F3", edgecolor="white", alpha=0.7)
    ax.axvline(per_image_dice.mean(), color="red", linestyle="--", label=f"Mean: {per_image_dice.mean():.3f}")
    ax.axvline(np.median(per_image_dice), color="green", linestyle="--", label=f"Median: {np.median(per_image_dice):.3f}")
    ax.set_xlabel("Dice Score")
    ax.set_ylabel("Count")
    ax.set_title("Per-Image Dice Distribution on Test Set")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "dice_histogram.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {WORK_DIR}/dice_histogram.png")
else:
    print("No checkpoint found. Run training first.")

---
## 9. Error Analysis

For a subset of images where Dice is lowest, analyze the failure modes.

In [ ]:
if BEST_CKPT and len(per_image_dice) > 0:
    # Find worst-performing samples
    if len(per_image_dice) < len(test_images):
        n_worst = min(4, len(test_images))
    else:
        n_worst = min(4, len(per_image_dice))
    
    worst_indices = np.argsort(per_image_dice)[:n_worst]
    
    print(f"Top {n_worst} worst Dice samples:")
    print("=" * 50)
    
    fig, axes = plt.subplots(n_worst, 4, figsize=(16, 4 * n_worst))
    if n_worst == 1:
        axes = axes.reshape(1, -1)
    
    for i, idx in enumerate(worst_indices):
        fname = test_images[idx]
        img_path = os.path.join(test_img_dir, fname)
        mask_path = os.path.join(test_mask_dir, fname)
        
        img_np = np.array(Image.open(img_path).convert("L"))
        mask_np = np.array(Image.open(mask_path).convert("L"))
        mask_bin = (mask_np > 127).astype(np.uint8)
        
        result = inference_model(model, img_path)
        pred = result.pred_sem_seg.data.cpu().numpy()
        pred_bin = (pred[0] > 0).astype(np.uint8)
        
        fn_rate = ((mask_bin == 1) & (pred_bin == 0)).sum() / mask_bin.sum() * 100 if mask_bin.sum() > 0 else 0
        fp_rate = ((mask_bin == 0) & (pred_bin == 1)).sum() / (mask_bin.size - mask_bin.sum()) * 100
        
        print(f"{i+1}. {fname[:20]:20s} | Dice: {per_image_dice[idx]:.4f} | FN: {fn_rate:.1f}% | FP: {fp_rate:.1f}%")
        
        # Error map
        fp = (pred_bin == 1) & (mask_bin == 0)
        fn = (pred_bin == 0) & (mask_bin == 1)
        tp = (pred_bin == 1) & (mask_bin == 1)
        
        error_map = np.zeros((*img_np.shape, 3), dtype=np.uint8)
        error_map[..., 0] = fp.astype(np.uint8) * 255
        error_map[..., 1] = tp.astype(np.uint8) * 255
        error_map[..., 2] = fn.astype(np.uint8) * 255
        
        axes[i][0].imshow(img_np, cmap="gray")
        axes[i][0].set_title(f"Input: {fname[:15]}", fontsize=9)
        axes[i][0].axis("off")
        
        axes[i][1].imshow(mask_bin, cmap="gray")
        axes[i][1].set_title(f"Ground Truth", fontsize=9)
        axes[i][1].axis("off")
        
        axes[i][2].imshow(pred_bin, cmap="gray")
        axes[i][2].set_title(f"Prediction (Dice={per_image_dice[idx]:.3f})", fontsize=9)
        axes[i][2].axis("off")
        
        axes[i][3].imshow(error_map)
        axes[i][3].set_title(f"Error: R=FP G=TP B=FN", fontsize=9)
        axes[i][3].axis("off")
    
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "error_analysis.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No predictions available for error analysis.")

---
## 10. Save & Persist Model Weights

Kaggle's `/kaggle/working/` is **ephemeral** — files are deleted when the session ends.
To keep your trained model, use ONE of the methods below.

### Option A: Kaggle Dataset (Recommended)
Push the checkpoint to a Kaggle Dataset. The Kaggle CLI is pre-installed.
Create a dataset once, then version it after each training run.

### Option B: Google Drive
Mount Drive and copy files there (requires authentication).

### Option C: Download from Output
Kaggle keeps notebook outputs available for ~20 days. Use the **Output** tab → **Download all**.

---


In [ ]:
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Save summary ----
summary_path = os.path.join(OUTPUT_DIR, "results_summary.txt")
with open(summary_path, "w") as f:
    f.write(f"NCCT Stroke Segmentation Results\n")
    f.write(f"Architecture: {ARCH_NAME}\n")
    f.write(f"Loss: CrossEntropyLoss (weighted) + DiceLoss\n")
    f.write(f"Class weights: [0.05, 1.0]\n")
    f.write(f"=" * 50 + "\n\n")
    
    if 'val_metrics' in dir() and val_metrics:
        best = max(val_metrics, key=lambda x: x["mDice"])
        final = val_metrics[-1]
        f.write(f"Best val mDice: {best['mDice']:.4f} (iter {best['iter']})\n")
        f.write(f"Best val mIoU:  {best['mIoU']:.4f} (iter {best['iter']})\n")
        f.write(f"Final val mDice: {final['mDice']:.4f} (iter {final['iter']})\n")
        f.write(f"\nValidation curve:\n")
        for m in val_metrics:
            f.write(f"  iter {m['iter']}: mDice={m['mDice']:.4f}  mIoU={m['mIoU']:.4f}\n")
    
    if 'per_image_dice' in dir() and len(per_image_dice) > 0:
        f.write(f"\nTest per-image Dice:\n")
        f.write(f"  Mean: {per_image_dice.mean():.4f}\n")
        f.write(f"  Median: {np.median(per_image_dice):.4f}\n")
        f.write(f"  Std: {per_image_dice.std():.4f}\n")

print(f"1. Summary saved: {summary_path}")

# ---- Copy key files to output ----
for src_name in ["validation_curves.png", "test_visualization.png", "dice_histogram.png", "error_analysis.png"]:
    src_path = os.path.join(WORK_DIR, src_name)
    if os.path.exists(src_path):
        shutil.copy2(src_path, OUTPUT_DIR)

if os.path.exists(PATCHED_CFG):
    shutil.copy2(PATCHED_CFG, os.path.join(OUTPUT_DIR, "config.py"))
print(f"2. Config and plots copied to {OUTPUT_DIR}")

# ---- Save best checkpoint ----
if 'BEST_CKPT' in dir() and BEST_CKPT:
    CKPT_NAME = f"{ARCH_NAME}_best.pth"
    ckpt_out = os.path.join(OUTPUT_DIR, CKPT_NAME)
    shutil.copy2(BEST_CKPT, ckpt_out)
    ckpt_size = os.path.getsize(ckpt_out) / 1e6
    print(f"3. Checkpoint saved: {CKPT_NAME} ({ckpt_size:.1f} MB)")
else:
    print(f"3. No checkpoint to save.")
    ckpt_out = None

print(f"\nAll outputs in: {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath) / 1e6
    print(f"  {f:30s} {size:.1f} MB")

### Persist Options

Choose ONE method below to keep your model permanently.

In [ ]:
# === OPTION A: Upload to Kaggle Dataset (Recommended) ===
# The Kaggle CLI is pre-installed on Kaggle notebooks.
# A Kaggle Dataset persists forever across sessions.
#
# USAGE:
# 1. Create a dataset (first run only):
#    !kaggle datasets create -p /kaggle/working/output \
#        --dir-mode tar --public \
#        -t "medical imaging" -t "brain CT" -t "stroke"
#
# 2. Update existing dataset (subsequent runs):
#    !kaggle datasets version -p /kaggle/working/output \
#        --dir-mode tar \
#        -m "{ARCH_NAME} - iter {best['iter']} - mDice {best['mDice']:.4f}"
#
# NOTE: You need a Kaggle API token (/kaggle_sets/kaggle.json or ~/.kaggle/kaggle.json)
# By default Kaggle provides this for the current user.

# Uncomment to run:
# !kaggle datasets version -p /kaggle/working/output --dir-mode tar \
#     -m "{ARCH_NAME} - iter {best['iter']} - mDice {best['mDice']:.4f}" \
#     2>&1 || echo "Kaggle dataset upload skipped (set up API key first)"

print('To persist: uncomment the kaggle datasets version command above.')
print('Your dataset name will be `kaggle datasets create` output.')

---
## 11. Full Schedule Training (Optional)

If you want to train longer (40K or 80K iterations) for better results, modify the cell below.

Typical values:
- 40K iterations: good balance of time and quality
- 80K iterations: best results, ~8-10 hours on 2× T4

In [ ]:
# Uncomment to retrain with longer schedule
# FULL_ITERS = 40000
# VAL_INTERVAL = 4000
# FULL_WORK_DIR = f"/kaggle/working/work_dirs/{ARCH_NAME}_full"
# 
# !cd {REPO_DIR} && torchrun --nproc_per_node={NUM_GPUS} tools/train.py {PATCHED_CFG} \
#     --work-dir={FULL_WORK_DIR} --amp --launcher=pytorch \
#     --cfg-options train_cfg.max_iters={FULL_ITERS} train_cfg.val_interval={VAL_INTERVAL} \
#     data_root={_abs_root}

---
## Summary

### What changed vs baseline:

| Aspect | Before | After |
|--------|--------|-------|
| Loss | CE only (class_weight=[0.1, 1.0]) | CE + Dice (class_weight=[0.05, 1.0]) |
| Weight ratio | 10:1 (stroke:bg) | 20:1 + Dice overlap loss |
| Augmentation | Resize + Flip + PhotoMetric | + RandomRotate 30° |
| Training | Single GPU | Multi-GPU DDP (2× T4) |
| Test output | Metrics only | Metrics + Per-image Dice + Visualizations + Error analysis |

### Expected improvements:
- Dice loss directly optimizes the evaluation metric
- Better class weights reduce false negatives (missed strokes)
- Enhanced augmentation improves generalization

### Outputs:
- `output/results_summary.txt`: All metrics
- `output/*.png`: Validation curves, test visualizations, error analysis, Dice histogram
- `output/{ARCH_NAME}_best.pth`: Trained model checkpoint